In [ ]:
# -*- coding: utf-8 -*-
"""
PARK (1999) "puro" no sentido clássico:
- alinhamento horizontal (shift) por N amostras (inteiro) em MALHA UNIFORME
- offset vertical (dS) por mínimos quadrados (média do erro)
- busca em grade do shift dentro do limite imposto pela sobreposição mínima
- suavização por média móvel

+ split térmico sem overlap
+ classificação multiclasse
+ prints completos
+ plots

IMPORTANTE sobre o erro FileNotFoundError:
- o arquivo 'base-completo--.pkl' não está no seu diretório atual.
  Use um caminho absoluto OU coloque o .pkl na mesma pasta do notebook.

Autor: Luiz Eduardo Abdala José (adaptado)
"""

import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

# ================================================================
# ===================== PARÂMETROS GERAIS ========================

# >>>>>>>>>>>>>>>>>> AJUSTE ISSO PARA EVITAR FileNotFoundError <<<<<<<<<<<<<<<<
# Opção 1 (recomendado): coloque o caminho absoluto:
# ARQ_BASE = r"C:\Users\SEU_USUARIO\Desktop\base-completo--.pkl"
# Opção 2: deixe só o nome, mas garanta que o .pkl está na mesma pasta do notebook.
ARQ_BASE = "base-completo--.pkl"
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

REF_TEMP = 30

# >>>>>>>>>>>>>>>>>> ALTERE APENAS AQUI <<<<<<<<<<<<<<<<<<<<<<
TEMP_DESEJADA = 78
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 70

# ------------------- PARK 1999 (grid search) ------------------
PARK_OVERLAP_MIN = 0.60   # sobreposição mínima (fração)
PARK_SMOOTH_WIN  = 5      # média móvel final
PARK_NSHIFTS     = 401    # número de shifts testados (ímpar recomendado)

# ------------------- RANDOM FOREST ----------------------------
RF_CLASSIF_PARAMS = dict(
    n_estimators=100,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.25,
    bootstrap=True,
    max_samples=0.70,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ================================================================
# ===================== FUNÇÕES AUXILIARES ======================
# ================================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    out = np.convolve(arr_pad, kernel, mode="valid")
    return out[:len(arr)]

def is_uniform_grid(fhz, rtol=1e-4, atol=1e-9):
    d = np.diff(fhz)
    return np.allclose(d, d[0], rtol=rtol, atol=atol)

def shift_by_samples(x, k):
    """
    Shift "clássico" por amostras (inteiro).
    k > 0: desloca para a direita (x fica "atrasado") -> preenche com borda
    k < 0: desloca para a esquerda -> preenche com borda
    """
    n = len(x)
    if k == 0:
        return x.copy()
    y = np.empty_like(x)
    if k > 0:
        y[:k] = x[0]
        y[k:] = x[:n-k]
    else:
        kk = -k
        y[n-kk:] = x[-1]
        y[:n-kk] = x[kk:]
    return y

# ================================================================
# ========================= PARK "PURO" ==========================
# ================================================================

def park_compensate_single_sampleshift(x, y_ref,
                                      overlap_min_frac=PARK_OVERLAP_MIN,
                                      smooth_win=PARK_SMOOTH_WIN,
                                      nshifts=PARK_NSHIFTS):
    """
    Procura k (inteiro) e dS (offset) que minimizam:
        Va(k) = sum_f [ y_ref(f) - (x_shift_k(f) + dS_k) ]^2
    onde:
        dS_k = mean( y_ref - x_shift_k )

    A janela de k é imposta pela sobreposição mínima:
        overlap_frac ≈ 1 - |k|/(n-1)
        exigir overlap_frac >= overlap_min_frac  -> |k| <= (1-overlap_min_frac)*(n-1)
    """
    n = len(x)
    if n < 5:
        return x.copy(), 0, 0.0

    k_max = int(np.floor((1.0 - overlap_min_frac) * (n - 1)))
    if k_max < 0:
        k_max = 0

    # grade de shifts inteiros simétrica
    # ex: nshifts=401 -> 401 valores de k entre -k_max e +k_max
    if k_max == 0:
        ks = np.array([0], dtype=int)
    else:
        ks = np.linspace(-k_max, +k_max, int(nshifts)).round().astype(int)
        ks = np.unique(ks)  # remove repetidos por arredondamento

    best_Va = np.inf
    best_k = 0
    best_dS = 0.0

    for k in ks:
        xs = shift_by_samples(x, int(k))
        dS = float(np.mean(y_ref - xs))
        r = (y_ref - (xs + dS))
        Va = float(np.sum(r * r))

        if Va < best_Va:
            best_Va = Va
            best_k = int(k)
            best_dS = float(dS)

    ycorr = shift_by_samples(x, best_k) + best_dS
    if smooth_win > 1:
        ycorr = moving_average(ycorr, smooth_win)

    return ycorr, best_k, best_dS

def park_batch_sampleshift(X, y_ref):
    Y = np.zeros_like(X)
    ks = np.zeros(len(X), dtype=int)
    dS = np.zeros(len(X), dtype=float)
    for i in range(len(X)):
        y, k, ds = park_compensate_single_sampleshift(X[i], y_ref)
        Y[i] = y
        ks[i] = k
        dS[i] = ds
    return Y, ks, dS

# ================================================================
# ============================ SCRIPT =============================
# ================================================================

timings = {}

# ----------- 0) CHECAGEM DE ARQUIVO (evita o erro) --------------
if not os.path.exists(ARQ_BASE):
    raise FileNotFoundError(
        f"Não encontrei '{ARQ_BASE}'.\n"
        f"- Coloque o arquivo .pkl na mesma pasta do notebook, OU\n"
        f"- Troque ARQ_BASE para o caminho absoluto (ex: r'C:\\\\...\\\\base-completo--.pkl').\n"
        f"Dica: no Jupyter, rode: import os; print(os.getcwd()) para ver a pasta atual."
    )

# ------------------ 1) CARREGA BASE -----------------------------
t0 = time.time()
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3
timings["load"] = time.time() - t0

# A versão por shift de amostras pressupõe malha aproximadamente uniforme
if not is_uniform_grid(fhz):
    raise ValueError(
        "A malha de frequência NÃO parece uniforme. "
        "O Park 'puro' por shift inteiro de amostras pode distorcer.\n"
        "Se quiser, eu adapto para shift por Hz via interpolação (mas aí não é 'puro')."
    )

# ------------------ 2) SPLIT TÉRMICO -----------------------------
t0 = time.time()
temps = sorted(df["temperatura_c"].unique())
temps_train = temps[::2]
temps_test  = temps[1::2]

df_train_raw = df[df["temperatura_c"].isin(temps_train)].copy()
df_test_raw  = df[df["temperatura_c"].isin(temps_test)].copy()
timings["split"] = time.time() - t0

# ------------------ 3) REFERÊNCIA (saudável) ---------------------
df_train_sem = df_train_raw[df_train_raw["falha"] == 0]
pool_ref = df_train_sem.loc[np.isclose(df_train_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

if len(pool_ref) > 0:
    y_ref = np.median(pool_ref, axis=0)
else:
    y_ref = np.median(df_train_sem[fcols].to_numpy(float), axis=0)

# ------------------ 4) PARK TREINO -------------------------------
t0 = time.time()
Xtr_raw = df_train_raw[fcols].to_numpy(float)
Y_train, k_tr, dS_tr = park_batch_sampleshift(Xtr_raw, y_ref)
timings["park_train"] = time.time() - t0

# ------------------ 5) PARK TESTE --------------------------------
t0 = time.time()
Xte_raw = df_test_raw[fcols].to_numpy(float)
Y_test, k_te, dS_te = park_batch_sampleshift(Xte_raw, y_ref)
timings["park_test"] = time.time() - t0

X_train = Y_train
X_test  = Y_test
y_train = df_train_raw["falha"].to_numpy(int)
y_test  = df_test_raw["falha"].to_numpy(int)

# ------------------ 6) CLASSIFICAÇÃO -----------------------------
t0 = time.time()
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS).fit(X_train, y_train)
timings["train_clf"] = time.time() - t0

t0 = time.time()
y_pred = clf.predict(X_test)
timings["predict_clf"] = time.time() - t0

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

# ------------------ 7) ESCOLHE CURVA PARA PLOTAR -----------------
df_plot = df_test_raw[df_test_raw["temperatura_c"] == TEMP_DESEJADA]
if len(df_plot) == 0:
    raise ValueError(
        f"NÃO existe curva de temperatura {TEMP_DESEJADA}°C no conjunto de teste!\n"
        f"Temperaturas no teste: {sorted(df_test_raw['temperatura_c'].unique())}"
    )

idx_show = df_plot.index[0]

# curva original (sem compensar)
y_orig = df.loc[idx_show, fcols].to_numpy(float)

# curva compensada (precisa achar a posição dela dentro do df_test_raw)
pos = int(np.where(df_test_raw.index.values == idx_show)[0][0])
y_comp = Y_test[pos]

# ================================================================
# =========================== PRINTS ==============================
# ================================================================

print("\n============== CLASSIFICAÇÃO (Park puro) ===================")
print("Matriz de confusão:")
print(cm)
print(f"\nACC = {acc:.4f}")
print(f"F1  = {macro_f1:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred, digits=4))

print("\n============== TEMPOS (s) ==================")
for k, v in timings.items():
    print(f"{k:20s}: {v:.4f}")

# ================================================================
# =========================== GRÁFICOS ============================
# ================================================================

plt.rcParams.update({
    "font.size": 12,
    "text.usetex": False,
    "font.family": "Times New Roman"
})

# (1) Gráfico padrão (maior)
plt.figure(figsize=(12, 6))
plt.plot(fhz_khz, y_ref, "--", c="black", lw=1.2, label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, y_orig, c="tab:red", lw=1.5, alpha=0.6, label=f"Original {TEMP_DESEJADA}°C")
plt.plot(fhz_khz, y_comp, c="tab:blue", lw=2.0, label=f"Compensado Park {TEMP_DESEJADA}°C")
plt.title(f"Compensação Park — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.grid(alpha=0)
plt.legend(frameon=True, facecolor="white", edgecolor="none")
plt.tight_layout()
plt.show()

# (2) Versão compacta (igual você tinha)
plt.rcParams.update({"font.size": 10})

plt.figure(figsize=(8, 4))
plt.plot(fhz_khz, y_ref, "--", c="black", lw=1.2, label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, y_orig, c="tab:red", lw=1.5, alpha=0.6, label=f"Original {TEMP_DESEJADA}°C")
plt.plot(fhz_khz, y_comp, c="tab:blue", lw=2.0, label=f"Compensado Park {TEMP_DESEJADA}°C")
plt.title(f"Compensação Park — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.grid(alpha=0)
plt.legend(frameon=True, facecolor="white", edgecolor="none")
plt.tight_layout()
plt.show()